# SKAB — Baseline Experiment

Random Forest and Isolation Forest with raw and temporal sensor features.


In [ ]:
from pathlib import Path
from datetime import datetime
import sys, json, time, hashlib, urllib.request, shutil, platform
from itertools import product
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, f1_score, average_precision_score, confusion_matrix
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

DATA = ROOT / 'data' / 'SKAB'
RUN = ROOT / 'results' / 'baseline' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
DATA.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)
SEEDS = [0, 1, 2]
SPLIT_SEED = 42
WINDOW = 30  # 최근 30개 관측값. 실제 시간 간격은 아래에서 확인
N_JOBS = 2  # 노트북 부하를 줄이기 위해 2개 작업 사용. 필요하면 1로 변경
COMMIT = 'b2c0d46c2971dcbfe71e26087b6d231998bb91c2'
SENSORS = ['Accelerometer1RMS','Accelerometer2RMS','Current','Pressure',
           'Temperature','Thermocouple','Voltage','Volume Flow RateRMS']
plt.rcParams.update({'figure.dpi':120, 'axes.spines.top':False, 'axes.spines.right':False})
print('Python:', sys.executable)
print('pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__)
print('작업 폴더:', ROOT)
print('이번 결과 폴더:', RUN)
print('CPU 기반 실험입니다. PyTorch/GPU는 사용하지 않습니다.')

## 1. Dataset Download and Inspection


In [ ]:
files = ([f'valve1/{i}.csv' for i in range(16)]
         + [f'valve2/{i}.csv' for i in range(4)]
         + [f'other/{i}.csv' for i in range(1,15)])
records, tables, download_manifest = [], {}, []
def download_file(fid):
    dest = DATA / fid
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        url = f'https://raw.githubusercontent.com/waico/SKAB/{COMMIT}/data/{fid}'
        last_error = None
        for attempt in range(3):
            try:
                request = urllib.request.Request(url, headers={'User-Agent':'SKAB-course-project'})
                with urllib.request.urlopen(request, timeout=60) as response:
                    payload = response.read()
                tmp = dest.with_suffix('.download')
                tmp.write_bytes(payload)
                tmp.replace(dest)
                last_error = None
                break
            except Exception as exc:
                last_error = exc
        if last_error is not None:
            raise RuntimeError(f'{fid} 다운로드 실패. 인터넷 연결 확인 후 이 셀을 다시 실행하세요.') from last_error
    return fid

with ThreadPoolExecutor(max_workers=4) as pool:
    tasks = [pool.submit(download_file, fid) for fid in files]
    for j, future in enumerate(as_completed(tasks), 1):
        future.result()
        if j % 10 == 0 or j == len(files): print(f'다운로드 준비: {j}/{len(files)}')

for j, fid in enumerate(files, 1):
    dest = DATA / fid
    df = pd.read_csv(dest, sep=';')
    required = SENSORS + ['datetime','anomaly','changepoint']
    if not set(required).issubset(df.columns):
        raise ValueError(f'{fid}의 열이 예상과 다릅니다: {df.columns.tolist()}')
    df['datetime'] = pd.to_datetime(df['datetime'], errors='raise')
    df = df.sort_values('datetime', kind='stable').reset_index(drop=True)
    if df['datetime'].duplicated().any():
        raise ValueError(f'{fid}: 중복 시각을 먼저 확인해야 합니다.')
    for col in SENSORS:
        df[col] = pd.to_numeric(df[col], errors='raise')
    if not df.anomaly.isin([0,1]).all():
        raise ValueError(f'{fid}: 정답 라벨 확인 필요')
    tables[fid] = df
    records.append({'file':fid,'family':fid.split('/')[0], 'rows':len(df),
                    'anomalies':int(df.anomaly.sum()),'anomaly_rate':df.anomaly.mean(),
                    'missing_sensors':int(df[SENSORS].isna().sum().sum()),
                    'median_interval_seconds':df.datetime.diff().dt.total_seconds().median()})
    download_manifest.append({'file':fid, 'sha256':hashlib.sha256(dest.read_bytes()).hexdigest()})
    if j % 10 == 0 or j == len(files): print(f'데이터 준비: {j}/{len(files)}')
inventory = pd.DataFrame(records)
inventory.to_csv(RUN/'data_inventory.csv',index=False,encoding='utf-8-sig')
print('총 파일:', len(inventory), '| 총 행:', inventory.rows.sum())
print('이상 비율:', f'{inventory.anomalies.sum()/inventory.rows.sum():.2%}')
print('센서 결측값:', inventory.missing_sensors.sum())
display(inventory.groupby('family').agg(files=('file','count'),rows=('rows','sum'),anomalies=('anomalies','sum')))
display(inventory[['family','median_interval_seconds']].groupby('family').describe())
display(next(iter(tables.values())).head())

## 2. File-Level Data Split


In [ ]:
train_ids, held_ids = train_test_split(inventory.file.tolist(), test_size=14, random_state=SPLIT_SEED, stratify=inventory.family)
lookup = inventory.set_index('file').family
val_ids, test_ids = train_test_split(held_ids, test_size=7, random_state=SPLIT_SEED, stratify=lookup.loc[held_ids])
assert not (set(train_ids)&set(val_ids) or set(train_ids)&set(test_ids) or set(val_ids)&set(test_ids))
split_map = {fid:split for split, ids in [('train',train_ids),('validation',val_ids),('test',test_ids)] for fid in ids}
inventory['split'] = inventory.file.map(split_map)
inventory.to_csv(RUN/'data_split.csv',index=False,encoding='utf-8-sig')
display(inventory.groupby(['split','family']).agg(files=('file','count'),rows=('rows','sum'),anomalies=('anomalies','sum')))

sensor_stats = pd.concat([d[SENSORS] for fid,d in tables.items() if fid in train_ids]).describe().T
sensor_stats.to_csv(RUN/'train_sensor_statistics.csv',encoding='utf-8-sig')
display(sensor_stats.round(3))
train_labeled = pd.concat([tables[fid] for fid in train_ids])
class_medians = train_labeled.groupby('anomaly')[SENSORS].median().T
class_medians.to_csv(RUN/'train_class_medians.csv',encoding='utf-8-sig')

example_id = sorted(train_ids)[0]  # 학습 파일 고정 예시, 성능을 보고 고르지 않음
example = tables[example_id]
fig, axs = plt.subplots(3,1,figsize=(10,6),sharex=True)
for ax, col in zip(axs, ['Pressure','Current','Volume Flow RateRMS']):
    t = np.arange(len(example))
    ax.plot(t,example[col],lw=1,label=col)
    ax.fill_between(t,0,1,where=example.anomaly.eq(1),transform=ax.get_xaxis_transform(),color='tomato',alpha=.18,label='Actual anomaly')
    ax.set_ylabel(col)
axs[0].set_title('Training example: '+example_id)
axs[-1].set_xlabel('Observation index within experiment')
fig.tight_layout(); fig.savefig(RUN/'data_example.png',dpi=180); plt.show()

## 3. Feature Engineering


In [ ]:
def build_features(df, mode):
    x = df[SENSORS].astype(float).replace([np.inf,-np.inf],np.nan).copy()
    if mode == 'raw': return x
    mean = x.rolling(WINDOW,min_periods=1).mean().add_suffix('_mean30')
    std = x.rolling(WINDOW,min_periods=1).std(ddof=0).add_suffix('_std30')
    delta = x.diff().add_suffix('_diff1')
    delta.iloc[0] = 0.0
    return pd.concat([x,mean,std,delta],axis=1)

def assemble(ids, mode):
    xx, yy, meta = [], [], []
    for fid in sorted(ids):
        df = tables[fid]
        xx.append(build_features(df,mode))
        yy.append(df.anomaly.to_numpy(dtype=int))
        meta.append(pd.DataFrame({'file':fid,'row':np.arange(len(df)), 'datetime':df.datetime.astype(str)}))
    return pd.concat(xx,ignore_index=True), np.concatenate(yy), pd.concat(meta,ignore_index=True)

packs={}
for mode in ['raw','temporal']:
    xtr,ytr,mtr = assemble(train_ids,mode)
    xva,yva,mva = assemble(val_ids,mode)
    imputer=SimpleImputer(strategy='median',keep_empty_features=True)
    packs[mode] = {'train':imputer.fit_transform(xtr),'val':imputer.transform(xva),
                   'ytrain':ytr,'yval':yva,'imputer':imputer,'columns':xtr.columns.tolist()}
    print(mode, '| 학습:',packs[mode]['train'].shape,'| 검증:',packs[mode]['val'].shape)
    assert set(np.unique(yva))=={0,1} and set(np.unique(ytr))=={0,1}
print('입력에서 제외된 열: datetime, anomaly, changepoint, file, row')

## 4. Hyperparameter and Threshold Selection


In [ ]:
def best_threshold(y, scores):
    p,r,t = precision_recall_curve(y,scores)
    f = np.divide(2*p[:-1]*r[:-1],p[:-1]+r[:-1],out=np.zeros_like(p[:-1]),where=(p[:-1]+r[:-1])>0)
    candidates = np.flatnonzero(np.isclose(f,f.max(),rtol=0,atol=1e-12))
    return float(t[candidates[-1]])

def metrics(y,scores,threshold):
    pred=(scores>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'precision':precision_score(y,pred,zero_division=0),
            'recall':recall_score(y,pred,zero_division=0),
            'f1':f1_score(y,pred,zero_division=0),
            'pr_auc_ap':average_precision_score(y,scores),
            'false_positive_rate':fp/(fp+tn) if fp+tn else np.nan,
            'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}

def fit_model(kind,mode,params,seed):
    p=packs[mode]
    if kind=='RF':
        model=RandomForestClassifier(n_estimators=200,random_state=seed,n_jobs=N_JOBS,**params)
        model.fit(p['train'],p['ytrain'])
        default_threshold=0.5
    else:
        model=IsolationForest(n_estimators=200,contamination='auto',random_state=seed,n_jobs=N_JOBS,**params)
        normal=p['train'][p['ytrain']==0]
        model.fit(normal)
        # IF 기준 임계값: 학습 정상 점수의 상위 1%. 검증 튜닝 전 사전 고정한 기준
        default_threshold=float(np.quantile(-model.score_samples(normal),0.99))
    return model,default_threshold

def anomaly_scores(model,kind,x):
    return model.predict_proba(x)[:,list(model.classes_).index(1)] if kind=='RF' else -model.score_samples(x)

grids={'RF':[{'max_depth':d,'min_samples_leaf':leaf,'class_weight':'balanced'} for d,leaf in product([12,None],[1,5])],
       'IF':[{'max_samples':n} for n in [256,1024]]}
search_rows=[]; chosen={}
started=time.perf_counter()
for kind in ['RF','IF']:
    for mode in ['raw','temporal']:
        options=[]
        for params in grids[kind]:
            tick=time.perf_counter()
            model,default_threshold=fit_model(kind,mode,params,SEEDS[0])
            score=anomaly_scores(model,kind,packs[mode]['val'])
            threshold=best_threshold(packs[mode]['yval'],score)
            row={'model':kind,'features':mode,'params':json.dumps(params),'threshold':threshold,
                 **metrics(packs[mode]['yval'],score,threshold),'fit_eval_seconds':time.perf_counter()-tick}
            search_rows.append(row); options.append((row,params))
            print(f"{kind:2s} {mode:8s} | 검증 F1 {row['f1']:.4f} | {row['fit_eval_seconds']:.1f}초 | {params}")
        winner=max(options,key=lambda v:v[0]['f1'])
        chosen[(kind,mode)]=winner[1]
search_df=pd.DataFrame(search_rows)
search_df.to_csv(RUN/'validation_search.csv',index=False,encoding='utf-8-sig')
print(f'설정 탐색 완료: {(time.perf_counter()-started)/60:.1f}분')
display(search_df.sort_values(['model','features','f1'],ascending=[True,True,False]).round(4))

## 5. Validation Across Training Seeds


In [ ]:
fitted={}; thresholds={}; default_thresholds={}; val_rows=[]
for kind in ['RF','IF']:
    for mode in ['raw','temporal']:
        for seed in SEEDS:
            tick=time.perf_counter()
            model,default_threshold=fit_model(kind,mode,chosen[(kind,mode)],seed)
            score=anomaly_scores(model,kind,packs[mode]['val'])
            threshold=best_threshold(packs[mode]['yval'],score)
            key=(kind,mode,seed)
            fitted[key]=model; thresholds[key]=threshold; default_thresholds[key]=default_threshold
            for rule,cut in [('fixed',default_threshold),('val_tuned',threshold)]:
                val_rows.append({'model':kind,'features':mode,'seed':seed,'rule':rule,'threshold':cut,
                                 **metrics(packs[mode]['yval'],score,cut)})
            print(f'{kind} {mode} seed{seed} 완료 ({time.perf_counter()-tick:.1f}초)')
val_detail=pd.DataFrame(val_rows)
val_detail.to_csv(RUN/'validation_metrics.csv',index=False,encoding='utf-8-sig')
val_summary=val_detail.query("rule=='val_tuned'").groupby(['model','features']).agg(
    f1_mean=('f1','mean'),f1_std=('f1','std'),precision_mean=('precision','mean'),
    recall_mean=('recall','mean'),pr_auc_ap_mean=('pr_auc_ap','mean')).reset_index()
val_summary=val_summary.sort_values('f1_mean',ascending=False,kind='stable')
val_summary.to_csv(RUN/'validation_selection.csv',index=False,encoding='utf-8-sig')
SELECTED=(val_summary.iloc[0]['model'],val_summary.iloc[0]['features'])
print('검증 F1 평균으로 선택한 모델·입력:', SELECTED)
display(val_summary.round(4))
print('아래 테스트 결과가 달라도 이 선택을 유지합니다.')

## 6. Test Evaluation


In [ ]:
test_packs={}
for mode in ['raw','temporal']:
    xt,yt,mt=assemble(test_ids,mode)
    test_packs[mode]={'x':packs[mode]['imputer'].transform(xt),'y':yt,'meta':mt}
test_rows=[]; saved_predictions=[]; score_store={}
for key,model in fitted.items():
    kind,mode,seed=key
    p=test_packs[mode]
    scores=anomaly_scores(model,kind,p['x'])
    score_store[key]=scores
    for rule,cut in [('fixed',default_thresholds[key]),('val_tuned',thresholds[key])]:
        test_rows.append({'model':kind,'features':mode,'seed':seed,'rule':rule,'threshold':cut,
                          **metrics(p['y'],scores,cut)})
    pred=p['meta'].copy()
    pred['y_true']=p['y']; pred['score']=scores
    pred['prediction']=(scores>=thresholds[key]).astype(int)
    pred['model']=kind; pred['features']=mode; pred['seed']=seed
    saved_predictions.append(pred)
test_detail=pd.DataFrame(test_rows)
test_detail.to_csv(RUN/'test_metrics.csv',index=False,encoding='utf-8-sig')
pd.concat(saved_predictions,ignore_index=True).to_csv(RUN/'test_predictions.csv.gz',index=False,compression='gzip')
METRICS=['precision','recall','f1','pr_auc_ap','false_positive_rate']
test_summary=test_detail.query("rule=='val_tuned'").groupby(['model','features'])[METRICS].agg(['mean','std'])
test_summary.columns=['_'.join(c) for c in test_summary.columns]
test_summary=test_summary.reset_index()
test_summary.to_csv(RUN/'test_summary.csv',index=False,encoding='utf-8-sig')
threshold_comparison=test_detail.groupby(['model','features','rule'])[METRICS].mean().reset_index()
threshold_comparison.to_csv(RUN/'threshold_comparison.csv',index=False,encoding='utf-8-sig')
print('최종 테스트 결과 — 검증에서 선택한 임계값 적용')
display(test_summary.round(4))
print('임계값 튜닝 전·후:')
display(threshold_comparison.round(4))
print('검증 기준 선택은 유지:', SELECTED)

## 7. Visualization and Anomaly Detection Examples


In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
labels=[f"{r.model} / {r.features}" for r in test_summary.itertuples()]
ax.bar(labels,test_summary.f1_mean,yerr=test_summary.f1_std,capsize=4,color=['#667f92','#237797','#b6a48b','#b7743c'])
ax.set_ylim(0,1.05); ax.set_ylabel('Test F1'); ax.set_title('Model and feature comparison')
fig.tight_layout();fig.savefig(RUN/'test_f1_comparison.png',dpi=200);plt.show()

example_test_id=sorted(test_ids)[0]
fig,axs=plt.subplots(4,1,figsize=(11,8),sharex=True,gridspec_kw={'height_ratios':[1,1,1,1]})
df=tables[example_test_id]; t=np.arange(len(df)); truth=df.anomaly.to_numpy()
axs[0].plot(t,df.Pressure,label='Pressure (bar)',color='#324b64',lw=1)
axs[0].set_ylabel('Pressure');axs[0].set_title('Test example: '+example_test_id+' | seed 0')
axs[1].step(t,truth,where='post',color='black',label='Actual anomaly');axs[1].set_ylabel('Actual');axs[1].set_ylim(-.1,1.2)
for ax,kind in zip(axs[2:],['RF','IF']):
    for mode,color in [('raw','#6b7f90'),('temporal','#c17c36')]:
        key=(kind,mode,SEEDS[0]); meta=test_packs[mode]['meta']; mask=meta.file.eq(example_test_id).to_numpy()
        # 임계값을 빼면 0 이상일 때 이상 판정. 모델 간 점수 크기 비교에는 사용하지 않음
        margin=score_store[key][mask]-thresholds[key]
        ax.plot(t,margin,label=mode,color=color,lw=1)
    ax.axhline(0,color='black',ls='--',lw=.8,label='Decision threshold')
    ax.fill_between(t,0,1,where=truth==1,transform=ax.get_xaxis_transform(),color='tomato',alpha=.12)
    ax.set_ylabel(kind+' score − threshold');ax.legend(loc='upper right',ncol=3,fontsize=8)
axs[-1].set_xlabel('Observation index within experiment')
fig.tight_layout();fig.savefig(RUN/'anomaly_example.png',dpi=200);plt.show()

per_file=[]
for key,scores in score_store.items():
    kind,mode,seed=key; p=test_packs[mode]
    for fid in sorted(test_ids):
        mask=p['meta'].file.eq(fid).to_numpy()
        per_file.append({'file':fid,'model':kind,'features':mode,'seed':seed,
                         **metrics(p['y'][mask],scores[mask],thresholds[key])})
pd.DataFrame(per_file).to_csv(RUN/'test_metrics_by_file.csv',index=False,encoding='utf-8-sig')

kind,mode=SELECTED
selected_model=fitted[(kind,mode,SEEDS[0])]
if kind=='RF':
    importance=pd.DataFrame({'feature':packs[mode]['columns'],'importance':selected_model.feature_importances_}).sort_values('importance',ascending=False)
    importance.to_csv(RUN/'rf_impurity_importance.csv',index=False,encoding='utf-8-sig')
    print('RF 불순도 기반 중요도: 인과관계로 해석하지 않음')
    display(importance.head(10))

## 8. Export Results


In [ ]:
manifest={'dataset':'SKAB v0.9, 34 labeled experiment files','source':'https://github.com/waico/SKAB',
          'commit':COMMIT,'data_sha256':download_manifest,'split_seed':SPLIT_SEED,
          'train_ids':sorted(train_ids),'validation_ids':sorted(val_ids),'test_ids':sorted(test_ids),
          'seeds':SEEDS,'window_observations':WINDOW,'feature_columns':{m:packs[m]['columns'] for m in packs},
          'search_seed':SEEDS[0],'selection_metric':'mean validation pointwise F1 across seeds',
          'selected_by_validation':{'model':SELECTED[0],'features':SELECTED[1]},
          'chosen_params':{kind+'_'+mode:params for (kind,mode),params in chosen.items()},
          'label_usage':{'RF':'train labels for classification; validation labels for parameter/threshold selection',
                         'IF':'train labels for filtering normal training rows; validation labels for parameter/threshold selection'},
          'threshold_baselines':{'RF':'0.5 probability','IF':'99th percentile of training normal anomaly scores'},
          'point_adjustment':False,'pr_auc_definition':'sklearn average_precision_score',
          'std_definition':'sample std across 3 training seeds (ddof=1), not confidence interval',
          'limitations':['single fixed file split','coarse family stratification, not unseen fault type evaluation',
                         'pooled pointwise metrics weight longer files more','no eventwise detection-delay metric',
                         'small seed-0 parameter search; no global-optimum claim'],
          'versions':{'python':platform.python_version(),'pandas':pd.__version__,'numpy':np.__version__,'sklearn':sklearn.__version__},
          'output_directory':str(RUN)}
(RUN/'experiment_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
notes="""RF: 정상/이상 라벨로 학습. IF: 학습 파일의 정상 라벨 행만 학습하므로 정상 선별에 라벨 사용.
두 모델 모두 검증 라벨로 설정/임계값 선택. 테스트로 설정을 재선택하지 않음.
Raw/Temporal 4조건 및 임계값 튜닝 전후는 사전에 정한 최종 비교.
최종 선택은 validation_selection.csv, 최종 성능은 test_summary.csv 확인.
3개 시드 평균/표준편차는 학습 난수 변동만 표현. Point adjustment 미적용.
PR-AUC는 Average Precision(AP) 방식. 특정 분할에서의 결과로 일반적인 최우수 모델을 단정하지 않음.
"""
(RUN/'READ_ME.txt').write_text(notes,encoding='utf-8-sig')
zip_path=Path(shutil.make_archive(str(RUN)+'_results','zip',root_dir=RUN))
print('모든 실험 완료!')
print('검증 기준 선택:',SELECTED)
print('Results archive:')
print(zip_path)
print(f'ZIP 크기: {zip_path.stat().st_size/1024/1024:.2f} MB')